<a href="https://colab.research.google.com/github/Raditya-Z/MachineLearning_244107020144/blob/main/JS-06%20-%20ANN/JS_06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas

In [1]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=fe04ec35e72411d5affebabafd1cdbadfb30c1a189419240c0b0dac7ce445fb2
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=75c8f4181e41d2802f6d9d392b7fa3fe5b3ec4b9bd08a41c4e47c5ad416504e7
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [3]:
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler

# -------------------
# Contoh dataset kecil untuk testing
# -------------------
np.random.seed(42)
n_samples = 10000   # jumlah database vector
d = 128             # dimensi
X = np.random.random((n_samples, d)).astype('float32')

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------
# Exact NN (brute-force)
# -------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------
# Annoy
# -------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------
# HNSW
# -------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------
# FAISS IVF
# -------------------
start = time.time()
quantizer = faiss.IndexFlatL2(d)
# Memperbaiki error dengan mengirimkan argumen secara posisional
index_faiss = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------
# Tampilkan ringkasan waktu
# -------------------
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")

Exact NN done in 1.928 s
Annoy done in 3.219 s
HNSW done in 9.417 s
FAISS IVF done in 0.367 s

=== Ringkasan Waktu (detik) ===
Exact NN : 1.928
Annoy    : 3.219
HNSW     : 9.417
FAISS    : 0.367


### Spesifikasi Smartphone yang Digunakan

> Pengujian menggunakan smartphone OPPO A77s dengan nomor model CPH2473. Perangkat ini menggunakan prosesor Qualcomm Snapdragon 680 Octa-core, RAM fisik 8 GB, dan penyimpanan internal 128 GB. Sistem operasi yang digunakan adalah Android 14 dengan antarmuka ColorOS 14.0. Pengujian pada perangkat Android dilakukan menggunakan aplikasi PyDroid 3 versi 8.6_arm64 untuk menjalankan program Python.

> Spesifikasi perangkat tersebut menjadi salah satu faktor yang dapat memengaruhi waktu komputasi, penggunaan memori, dan keberhasilan menjalankan algoritma Exact Nearest Neighbor, Annoy, HNSWLIB, serta FAISS. Hasil pengujian kemudian dibandingkan dengan hasil yang diperoleh melalui Google Colab.

# 2. Analisis
Berdasarkan hasil percobaan, FAISS IVF memiliki waktu komputasi paling singkat di Google Colab, yaitu 0,367 detik. Sementara itu, Exact NN membutuhkan 1,928 detik, Annoy 3,219 detik, dan HNSW 9,417 detik.

Pada OPPO A77s, Exact NN menjadi metode tercepat dengan waktu 1,223 detik, disusul Annoy selama 1,957 detik dan HNSW selama 7,538 detik. Dalam percobaan ini, waktu komputasi pada HP lebih singkat daripada Google Colab untuk ketiga metode yang berhasil dijalankan. Namun, hasil ini tidak berarti smartphone secara umum lebih cepat daripada Colab, karena waktu pengukuran dapat dipengaruhi kondisi runtime, penggunaan sumber daya, dan proses persiapan indeks yang ikut dihitung.

FAISS IVF belum dapat dibandingkan secara langsung pada HP karena instalasinya gagal. Selain itu, pengukuran ini mencakup proses yang berbeda pada masing-masing metode, sehingga belum sepenuhnya menggambarkan waktu pencarian saja.

# 3. Kesimpulan
Percobaan menunjukkan bahwa performa setiap metode berbeda pada kedua lingkungan. FAISS IVF menjadi yang tercepat di Google Colab, sedangkan Exact NN menjadi yang tercepat di OPPO A77s untuk pengujian ini. Annoy dan HNSW juga berhasil dijalankan di HP, tetapi membutuhkan waktu lebih lama dibandingkan Exact NN. Kendala instalasi FAISS di Android menjadi keterbatasan dalam perbandingan. Hasil ini berlaku untuk dataset 10.000 vektor berdimensi 128 dan tidak otomatis berlaku untuk dataset yang lebih besar.